# N-gram Language Modeling and LSTM for Protein Sequence Prediction

This notebook looks at the same problem — predicting the next amino acid in a protein sequence — from two different angles:

1. A classic **statistical approach**: unigram, bigram, and trigram language models built directly from amino acid frequencies, with add-α smoothing.
2. A **neural approach**: an LSTM-based sequence model trained in TensorFlow/Keras.

Both approaches are evaluated on the same held-out test set (perplexity for the N-gram models, cross-entropy and accuracy for the LSTM), so the results can be compared directly.

Protein sequences behave a lot like language: amino acids don't appear in a random order, and the identity of nearby residues carries information about what's likely to come next. That similarity is what makes N-gram models — originally built for text — a reasonable baseline here.


## 1. Data

Sequences come from **UniProt**, for three species: human, mouse, and rat (`humans.fasta`, `mouse.fasta`, `rat.fasta`). Only the raw amino acid sequences are used — headers and metadata are discarded during loading.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Data paths - point these at your local copies of the UniProt FASTA files
DATA_DIR = "./data"

def load_protein_sequences(file_path):
    """Read a FASTA file and return just the sequence lines, skipping
    the '>' header lines."""
    with open(file_path, 'r') as file:
        sequences = [line.strip() for line in file if line.strip() and not line.startswith('>')]
    return sequences

human_seqs = load_protein_sequences(f"{DATA_DIR}/humans.fasta")
mouse_seqs = load_protein_sequences(f"{DATA_DIR}/mouse.fasta")
rat_seqs = load_protein_sequences(f"{DATA_DIR}/rat.fasta")

sequences = human_seqs + mouse_seqs + rat_seqs

# The 20 standard amino acids
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'


## 2. Preprocessing

Two cleanup steps here: drop any character that isn't one of the 20 standard amino acids (the raw FASTA files can contain ambiguity codes or formatting artifacts), and drop sequences that are too short or too long to be useful for training.


In [ ]:
def improved_preprocessing(sequences):
    """Keep only valid amino acid characters, and drop sequences outside
    a reasonable length range (too short to be informative, or too long
    to train on efficiently)."""
    valid_sequences = []
    for seq in sequences:
        cleaned_sequences = ''.join([aa for aa in seq if aa in amino_acids])
        if 10 <= len(cleaned_sequences) <= 1000:
            valid_sequences.append(cleaned_sequences)
    return valid_sequences

sequences = improved_preprocessing(sequences)

print(f"Total protein sequences loaded: {len(sequences)}")
print("Sample of the first 10 sequences:")
for i, seq in enumerate(sequences[:10]):
    print(f"Sequence {i+1}: length {len(seq)} amino acids")


## 3. Train / Test Split

In [ ]:
np.random.seed(42)
np.random.shuffle(sequences)

train_size = int(0.8 * len(sequences))
train_sequences = sequences[:train_size]
test_sequences = sequences[train_size:]

train_sequences = improved_preprocessing(train_sequences)
test_sequences = improved_preprocessing(test_sequences)

print("Split summary:")
print(f"Train sequences: {len(train_sequences)} ({len(train_sequences)/len(sequences)*100:.1f}%)")
print(f"Test sequences: {len(test_sequences)} ({len(test_sequences)/len(sequences)*100:.1f}%)")


In [ ]:
def calculate_sequence_stats(sequences):
    lengths = [len(seq) for seq in sequences]
    return {
        'mean_length': np.mean(lengths),
        'min_length': np.min(lengths),
        'max_length': np.max(lengths),
        'median_length': np.median(lengths)
    }

train_stats = calculate_sequence_stats(train_sequences)
test_stats = calculate_sequence_stats(test_sequences)

print("Train sequence length stats:")
print(f"Mean: {train_stats['mean_length']:.1f}  Min: {train_stats['min_length']}  Max: {train_stats['max_length']}")

print("\nTest sequence length stats:")
print(f"Mean: {test_stats['mean_length']:.1f}  Min: {test_stats['min_length']}  Max: {test_stats['max_length']}")


## 4. N-gram Language Models

Three probabilistic models, each conditioning on more context than the last:

- **Unigram**: just the overall frequency of each amino acid, no context.
- **Bigram**: probability of the next amino acid given the previous one.
- **Trigram**: probability of the next amino acid given the previous two.


In [ ]:
def calculate_unigram(sequences):
    """P(amino acid) - overall frequency, no context."""
    counts = {aa: 0 for aa in amino_acids}
    total = 0

    for seq in sequences:
        for aa in seq:
            if aa in counts:
                counts[aa] += 1
                total += 1

    probabilities = {aa: count / total for aa, count in counts.items()}
    return probabilities


def calculate_bigram(sequences):
    """P(next amino acid | previous amino acid)."""
    counts = defaultdict(lambda: defaultdict(int))
    context_counts = defaultdict(int)

    for seq in sequences:
        for i in range(len(seq) - 1):
            current = seq[i]
            next_aa = seq[i + 1]
            if current in amino_acids and next_aa in amino_acids:
                counts[current][next_aa] += 1
                context_counts[current] += 1

    probabilities = defaultdict(dict)
    for aa1 in amino_acids:
        for aa2 in amino_acids:
            if context_counts[aa1] > 0:
                probabilities[aa1][aa2] = counts[aa1][aa2] / context_counts[aa1]
            else:
                probabilities[aa1][aa2] = 0.0

    return probabilities


def calculate_trigram(sequences):
    """P(next amino acid | previous two amino acids)."""
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    context_counts = defaultdict(lambda: defaultdict(int))

    for seq in sequences:
        for i in range(len(seq) - 2):
            aa1, aa2, aa3 = seq[i], seq[i + 1], seq[i + 2]
            if aa1 in amino_acids and aa2 in amino_acids and aa3 in amino_acids:
                counts[aa1][aa2][aa3] += 1
                context_counts[aa1][aa2] += 1

    probabilities = defaultdict(lambda: defaultdict(dict))
    for aa1 in amino_acids:
        for aa2 in amino_acids:
            for aa3 in amino_acids:
                if context_counts[aa1][aa2] > 0:
                    probabilities[aa1][aa2][aa3] = counts[aa1][aa2][aa3] / context_counts[aa1][aa2]
                else:
                    probabilities[aa1][aa2][aa3] = 0.0

    return probabilities


### Visualizing the Unigram Distribution

In [ ]:
unigram_train = calculate_unigram(train_sequences)

def plot_unigram(probabilities):
    amino_acids_sorted = sorted(probabilities.keys())
    probs = [probabilities[aa] for aa in amino_acids_sorted]

    plt.figure(figsize=(12, 6))
    plt.bar(amino_acids_sorted, probs)
    plt.title('Unigram Amino Acid Probabilities')
    plt.xlabel('Amino Acid')
    plt.ylabel('Probability')
    plt.show()

plot_unigram(unigram_train)


### Visualizing the Bigram Distribution

In [ ]:
bigram_train = calculate_bigram(train_sequences)

def plot_bigram(probabilities):
    amino_acids_sorted = sorted(probabilities.keys())
    matrix = np.zeros((len(amino_acids_sorted), len(amino_acids_sorted)))

    for i, aa1 in enumerate(amino_acids_sorted):
        for j, aa2 in enumerate(amino_acids_sorted):
            matrix[i, j] = probabilities[aa1][aa2]

    plt.figure(figsize=(12, 10))
    sns.heatmap(matrix, xticklabels=amino_acids_sorted, yticklabels=amino_acids_sorted,
                cmap='viridis', annot=True, fmt='.2f')
    plt.title('Bigram Conditional Probabilities')
    plt.xlabel('Next Amino Acid')
    plt.ylabel('Current Amino Acid')
    plt.show()

plot_bigram(bigram_train)


### Visualizing a Few Trigram Distributions

Plotting all trigram combinations would be unreadable, so this shows the distribution for a handful of two-amino-acid contexts as an example.

In [ ]:
trigram_train = calculate_trigram(train_sequences)

def plot_sample_trigrams(probabilities, sample_pairs):
    amino_acids_sorted = sorted(probabilities.keys())
    num_samples = len(sample_pairs)

    fig, axes = plt.subplots(num_samples, 1, figsize=(10, 4 * num_samples))
    if num_samples == 1:
        axes = [axes]

    for idx, (aa1, aa2) in enumerate(sample_pairs):
        probs = [probabilities[aa1][aa2][aa3] for aa3 in amino_acids_sorted]
        bars = axes[idx].bar(amino_acids_sorted, probs, color='skyblue', edgecolor='black', width=0.7)

        for bar in bars:
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width() / 2, height,
                            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

        axes[idx].set_title(f'Probability after {aa1}{aa2}', fontsize=12, pad=10)
        axes[idx].set_xlabel('Next amino acid', fontsize=10)
        axes[idx].set_ylabel('Probability', fontsize=10)
        axes[idx].set_ylim(0, max(probs) * 1.2)
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

sample_pairs = [('A', 'L'), ('V', 'E'), ('G', 'H')]
plot_sample_trigrams(trigram_train, sample_pairs)


## 5. Add-α Smoothing and Perplexity

Raw counts give a probability of exactly zero to any amino acid combination that never showed up in training, which breaks down the moment the test set contains something new. Add-α smoothing fixes that by adding a small constant to every count before normalizing.

Perplexity is used to compare the three models on the test set - lower is better. It's the standard way to score a language model: roughly, "how surprised is the model, on average, by the next token?"


In [ ]:
def calculate_smoothed_models(sequences, alpha=1.0):
    """Rebuild the unigram/bigram/trigram models with add-alpha smoothing
    so no valid combination ever gets a probability of exactly zero."""

    unigram_counts = defaultdict(int)
    for seq in sequences:
        for aa in seq:
            if aa in amino_acids:
                unigram_counts[aa] += 1
    total = sum(unigram_counts.values())
    unigram_model = {aa: (count + alpha) / (total + alpha * len(amino_acids))
                      for aa, count in unigram_counts.items()}

    bigram_counts = defaultdict(lambda: defaultdict(int))
    context_counts = defaultdict(int)
    for seq in sequences:
        for i in range(len(seq) - 1):
            a, b = seq[i], seq[i + 1]
            if a in amino_acids and b in amino_acids:
                bigram_counts[a][b] += 1
                context_counts[a] += 1
    bigram_model = {a: {b: (bigram_counts[a][b] + alpha) / (context_counts[a] + alpha * len(amino_acids))
                         for b in bigram_counts[a]}
                     for a in bigram_counts}

    trigram_counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    pair_counts = defaultdict(lambda: defaultdict(int))
    for seq in sequences:
        for i in range(len(seq) - 2):
            a, b, c = seq[i], seq[i + 1], seq[i + 2]
            if a in amino_acids and b in amino_acids and c in amino_acids:
                trigram_counts[a][b][c] += 1
                pair_counts[a][b] += 1
    trigram_model = {a: {b: {c: (trigram_counts[a][b][c] + alpha) / (pair_counts[a][b] + alpha * len(amino_acids))
                              for c in trigram_counts[a][b]}
                          for b in trigram_counts[a]}
                      for a in trigram_counts}

    return unigram_model, bigram_model, trigram_model


def calculate_perplexity_all(models, test_sequences):
    """Compute perplexity on the test set for each of the three models."""
    results = {}
    unigram, bigram, trigram = models

    for name, model, n in [('unigram', unigram, 1), ('bigram', bigram, 2), ('trigram', trigram, 3)]:
        log_prob_sum = 0
        total = 0

        for seq in test_sequences:
            for i in range(len(seq) - n):
                if name == 'unigram':
                    prob = model.get(seq[i + 1], 1 / (len(amino_acids) + 1))
                elif name == 'bigram':
                    prob = model.get(seq[i], {}).get(seq[i + 1], 1 / (len(amino_acids) + 1))
                elif name == 'trigram':
                    prob = model.get(seq[i], {}).get(seq[i + 1], {}).get(seq[i + 2], 1 / (len(amino_acids) + 1))

                log_prob_sum += np.log(prob + 1e-10)
                total += 1

        perplexity = np.exp(-log_prob_sum / total) if total > 0 else float('inf')
        results[name] = perplexity

    return results


unigram_sm, bigram_sm, trigram_sm = calculate_smoothed_models(train_sequences, alpha=0.5)
results = calculate_perplexity_all((unigram_sm, bigram_sm, trigram_sm), test_sequences)

print("Perplexity on the test set:")
print(f"Unigram: {results['unigram']:.2f}")
print(f"Bigram:  {results['bigram']:.2f}")
print(f"Trigram: {results['trigram']:.2f}")


## 6. LSTM Neural Model

The N-gram models above only ever look at the last one or two amino acids. An LSTM can, in principle, use the whole preceding sequence, so it's a natural next step to compare against.

Each sequence is turned into a set of "prefix -> next token" training pairs (e.g. for `MKV`, the pairs are `M -> K` and `MK -> V`), padded to a fixed length, and fed to a small embedding + LSTM + dense-softmax model.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}
vocab_size = len(amino_acids)

def build_nn_model(vocab_size=20, embedding_dim=16, lstm_units=64):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim),
        LSTM(lstm_units),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
def prepare_nn_data(sequences, max_len=100):
    """Turn each sequence into (prefix, next_token) pairs, then pad the
    prefixes to a fixed length so they can be batched together."""
    X, y = [], []
    for seq in sequences:
        seq_indices = [aa_to_idx[aa] for aa in seq[:max_len]]
        for i in range(1, len(seq_indices)):
            X.append(seq_indices[:i])
            y.append(seq_indices[i])

    X = pad_sequences(X, maxlen=max_len - 1, padding='pre')
    y = to_categorical(y, num_classes=vocab_size)
    return X, y


X_train, y_train = prepare_nn_data(train_sequences)
X_test, y_test = prepare_nn_data(test_sequences)

nn_model = build_nn_model()
history = nn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))


## 7. LSTM Evaluation

Cross-entropy and accuracy on the same test set used for the N-gram perplexity numbers.


In [ ]:
import numpy as np
from tensorflow.keras.metrics import CategoricalCrossentropy

def evaluate_neural_network(model, test_sequences):
    X_test, y_test = prepare_nn_data(test_sequences)

    cce = CategoricalCrossentropy()
    cross_entropy = cce(y_test, model.predict(X_test)).numpy()

    y_pred = model.predict(X_test)
    accuracy = np.mean(np.argmax(y_test, axis=1) == np.argmax(y_pred, axis=1))

    return cross_entropy, accuracy


nn_cross_entropy, nn_accuracy = evaluate_neural_network(nn_model, test_sequences)

print("LSTM evaluation:")
print(f"Cross-entropy: {nn_cross_entropy:.4f}")
print(f"Accuracy: {nn_accuracy:.4f}")


### Results

| Model | Metric | Score |
|---|---|---|
| Unigram | Perplexity | 17.93 |
| Bigram | Perplexity | 17.72 |
| Trigram | Perplexity | 17.17 |
| LSTM | Cross-entropy | 2.70 |
| LSTM | Accuracy | 0.169 |

A few honest observations:

- Perplexity improves slightly as more context is added (unigram → bigram → trigram), which makes sense: neighboring amino acids do carry some information, but not a huge amount — protein sequences are much less locally predictable than natural language.
- The LSTM's ~16.9% accuracy looks low at first glance, but with 20 possible amino acids, random guessing would score about 5%. So the model is learning a real signal, it's just a weak one - which matches the modest gains seen across the N-gram models too. Next-amino-acid prediction from short local context is a genuinely hard problem; most of what determines a protein's sequence isn't visible from a short local window.
- Given more training data, a larger model, or additional biological features (e.g. secondary structure), the neural model would likely have more room to improve than the N-gram approach, since it isn't capped at a fixed context window.


## 8. Interactive Prediction

A small utility to try the bigram and trigram models interactively: enter one or two amino acids and see the model's predicted probabilities for what comes next.


In [ ]:
def predict_next_amino_acids():
    aa1 = input(f"Enter the first amino acid (one of {amino_acids}): ").upper()
    while aa1 not in amino_acids:
        aa1 = input("Invalid amino acid, try again: ").upper()

    print("\nBigram model prediction for the next amino acid:")
    probs = bigram_train[aa1]
    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    for aa, prob in sorted_probs:
        print(f"{aa}: {prob:.4f}")

    aa2 = input("\nEnter the second amino acid: ").upper()
    while aa2 not in amino_acids:
        aa2 = input("Invalid amino acid, try again: ").upper()

    print(f"\nTrigram model prediction for the sequence {aa1}{aa2}:")
    probs = trigram_train[aa1][aa2]
    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    for aa, prob in sorted_probs:
        print(f"{aa}: {prob:.4f}")


predict_next_amino_acids()
